# Feature Engineering — bureau.csv + bureau_balance.csv

**¿Qué es bureau.csv?**
Contiene los créditos que cada cliente tiene reportados en el buró de crédito externo
(equivalente a Datacrédito o TransUnion en Colombia). Cada fila es un crédito diferente
del mismo cliente — por eso hay 1.7M de filas para 307k clientes.

**¿Qué es bureau_balance.csv?**
El estado mensual de cada crédito del buró. Para cada crédito en bureau.csv,
hay N filas aquí (una por mes). Con 27M de filas es la tabla más grande del dataset.

**Objetivo:** colapsar estas dos tablas en UNA SOLA FILA por cliente con features
que capturen el comportamiento histórico de crédito: cuántos créditos tuvo,
si pagó a tiempo, cuánta deuda tiene activa, etc.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
DATA_PATH = '../data/raw/'

print('Librerías cargadas.')

Librerías cargadas.


## 1. Exploración rápida de bureau.csv

In [2]:
bureau = pd.read_csv(DATA_PATH + 'bureau.csv')
print(f'Shape: {bureau.shape}')
print(f'\nColumnas: {bureau.columns.tolist()}')
print(f'\nClientes únicos: {bureau["SK_ID_CURR"].nunique():,}')
print(f'Créditos por cliente (promedio): {len(bureau) / bureau["SK_ID_CURR"].nunique():.1f}')
print(f'\nMissings:')
print(bureau.isnull().mean().round(3)[bureau.isnull().mean() > 0])

Shape: (1716428, 17)

Columnas: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']

Clientes únicos: 305,811
Créditos por cliente (promedio): 5.6

Missings:
DAYS_CREDIT_ENDDATE       0.061
DAYS_ENDDATE_FACT         0.369
AMT_CREDIT_MAX_OVERDUE    0.655
AMT_CREDIT_SUM            0.000
AMT_CREDIT_SUM_DEBT       0.150
AMT_CREDIT_SUM_LIMIT      0.345
AMT_ANNUITY               0.715
dtype: float64


## 2. Features desde bureau.csv

Agrupamos por SK_ID_CURR y calculamos agregaciones que capturan
el comportamiento histórico de crédito de cada cliente.

In [3]:
# Agregaciones numéricas
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    # Cantidad de créditos
    BUREAU_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),

    # Deuda y crédito
    BUREAU_AMT_CREDIT_SUM=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AMT_CREDIT_SUM_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_AMT_CREDIT_MAX_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),

    # Mora
    BUREAU_DAYS_CREDIT_MEAN=('DAYS_CREDIT', 'mean'),
    BUREAU_DAYS_CREDIT_ENDDATE_MEAN=('DAYS_CREDIT_ENDDATE', 'mean'),
    BUREAU_DAYS_OVERDUE_MEAN=('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_DAYS_OVERDUE_MAX=('CREDIT_DAY_OVERDUE', 'max'),

    # Prolongaciones (cuántas veces extendieron el crédito)
    BUREAU_CNT_PROLONG_SUM=('CNT_CREDIT_PROLONG', 'sum'),
).reset_index()

# Feature derivada: ratio deuda/crédito total (qué porcentaje de su crédito está en deuda)
bureau_agg['BUREAU_DEBT_RATIO'] = (
    bureau_agg['BUREAU_AMT_CREDIT_SUM_DEBT'] /
    (bureau_agg['BUREAU_AMT_CREDIT_SUM'] + 1)  # +1 evita división por cero
)

print(f'Shape bureau_agg: {bureau_agg.shape}')
print(f'Features generadas: {bureau_agg.shape[1] - 1}')  # -1 por SK_ID_CURR
bureau_agg.head(3)

Shape bureau_agg: (305811, 14)
Features generadas: 13


,SK_ID_CURR,BUREAU_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_AMT_CREDIT_SUM,BUREAU_AMT_CREDIT_SUM_DEBT,BUREAU_AMT_CREDIT_MAX_OVERDUE,BUREAU_AMT_ANNUITY_MEAN,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_OVERDUE_MEAN,BUREAU_DAYS_OVERDUE_MAX,BUREAU_CNT_PROLONG_SUM,BUREAU_DEBT_RATIO
0,100001,7,3,4,1453365.000,596686.5,NaN,3545.357143,-735.00,82.428571,0.0,0,0,0.410555
1,100002,8,2,6,865055.565,245781.0,5043.645,0.000000,-874.00,-349.000000,0.0,0,0,0.284121
2,100003,4,1,3,1017400.500,0.0,0.000,NaN,-1400.75,-544.500000,0.0,0,0,0.000000


## 3. Features desde bureau_balance.csv

bureau_balance tiene el estado mensual de cada crédito (C=corriente, X=sin info,
0-5=días de mora). Con 27M de filas lo procesamos agrupando primero por crédito
y luego por cliente.

In [4]:
bureau_balance = pd.read_csv(DATA_PATH + 'bureau_balance.csv')
print(f'Shape: {bureau_balance.shape}')
print(f'STATUS únicos: {bureau_balance["STATUS"].unique()}')
print(f'Créditos únicos: {bureau_balance["SK_ID_BUREAU"].nunique():,}')

# Paso 1: agregar por crédito (SK_ID_BUREAU)
# STATUS: C=corriente, X=sin info, 0=sin mora, 1-5=meses de mora
bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(
    BB_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),
    BB_STATUS_OVERDUE_COUNT=('STATUS', lambda x: x.isin(['1','2','3','4','5']).sum()),
    BB_STATUS_C_COUNT=('STATUS', lambda x: (x == 'C').sum()),
).reset_index()

bb_agg['BB_OVERDUE_RATIO'] = bb_agg['BB_STATUS_OVERDUE_COUNT'] / (bb_agg['BB_MONTHS_COUNT'] + 1)

# Paso 2: unir con bureau para tener SK_ID_CURR y agregar por cliente
bb_with_curr = bureau[['SK_ID_CURR', 'SK_ID_BUREAU']].merge(bb_agg, on='SK_ID_BUREAU', how='left')

bureau_balance_agg = bb_with_curr.groupby('SK_ID_CURR').agg(
    BB_MONTHS_COUNT_MEAN=('BB_MONTHS_COUNT', 'mean'),
    BB_OVERDUE_COUNT_SUM=('BB_STATUS_OVERDUE_COUNT', 'sum'),
    BB_OVERDUE_RATIO_MEAN=('BB_OVERDUE_RATIO', 'mean'),
    BB_OVERDUE_RATIO_MAX=('BB_OVERDUE_RATIO', 'max'),
).reset_index()

print(f'\nShape bureau_balance_agg: {bureau_balance_agg.shape}')
bureau_balance_agg.head(3)

Shape: (27299925, 3)
STATUS únicos: ['C' '0' 'X' '1' '2' '3' '5' '4']
Créditos únicos: 817,395

Shape bureau_balance_agg: (305811, 5)


,SK_ID_CURR,BB_MONTHS_COUNT_MEAN,BB_OVERDUE_COUNT_SUM,BB_OVERDUE_RATIO_MEAN,BB_OVERDUE_RATIO_MAX
0,100001,24.571429,1.0,0.007143,0.05
1,100002,13.750000,27.0,0.231905,0.40
2,100003,NaN,0.0,NaN,NaN


## 4. Combinar bureau + bureau_balance en una sola tabla

In [5]:
bureau_features = bureau_agg.merge(bureau_balance_agg, on='SK_ID_CURR', how='left')

print(f'Shape final bureau_features: {bureau_features.shape}')
print(f'Clientes con features de bureau: {len(bureau_features):,}')
print(f'Features totales: {bureau_features.shape[1] - 1}')
print(f'\nMissings:')
print(bureau_features.isnull().mean().round(3)[bureau_features.isnull().mean() > 0])

# Guardar para usarlo en el notebook final
bureau_features.to_parquet('../data/processed/bureau_features.parquet', index=False)
print('\nGuardado en data/processed/bureau_features.parquet')

Shape final bureau_features: (305811, 18)
Clientes con features de bureau: 305,811
Features totales: 17

Missings:
BUREAU_AMT_CREDIT_MAX_OVERDUE      0.304
BUREAU_AMT_ANNUITY_MEAN            0.613
BUREAU_DAYS_CREDIT_ENDDATE_MEAN    0.008
BB_MONTHS_COUNT_MEAN               0.560
BB_OVERDUE_RATIO_MEAN              0.560
BB_OVERDUE_RATIO_MAX               0.560
dtype: float64

Guardado en data/processed/bureau_features.parquet


## 5. Validación — ¿estas features predicen el incumplimiento?

Unimos con application_train para ver si las features nuevas tienen poder predictivo.

In [6]:
app = pd.read_csv(DATA_PATH + 'application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
val = app.merge(bureau_features, on='SK_ID_CURR', how='left')

print('Correlación con TARGET (top features de bureau):')
corr = val.drop(columns='SK_ID_CURR').corr()['TARGET'].drop('TARGET')
print(corr.abs().sort_values(ascending=False).head(10).round(4))

print(f'\nClientes SIN historial en bureau: {val["BUREAU_COUNT"].isna().sum():,} ({val["BUREAU_COUNT"].isna().mean()*100:.1f}%)')
print('→ Para estos clientes todas las features de bureau serán NaN — LightGBM lo maneja nativo.')

Correlación con TARGET (top features de bureau):
BUREAU_DAYS_CREDIT_MEAN            0.0897
BB_MONTHS_COUNT_MEAN               0.0802
BUREAU_ACTIVE_COUNT                0.0671
BB_OVERDUE_RATIO_MEAN              0.0582
BB_OVERDUE_RATIO_MAX               0.0557
BUREAU_DAYS_CREDIT_ENDDATE_MEAN    0.0470
BUREAU_CLOSED_COUNT                0.0308
BB_OVERDUE_COUNT_SUM               0.0172
BUREAU_AMT_CREDIT_SUM              0.0141
BUREAU_DAYS_OVERDUE_MEAN           0.0081
Name: TARGET, dtype: float64

Clientes SIN historial en bureau: 44,020 (14.3%)
→ Para estos clientes todas las features de bureau serán NaN — LightGBM lo maneja nativo.
